# Results figures and appendix tables (thesis Chapter 4, Appendix B)

This notebook produces the figures of the results chapter and several appendix tables
from existing result files only: `results/main/perc97/all_iterations.csv` and the
paired-Wilcoxon table next to it (`scripts/paired_wilcoxon_bh.py`), the per-window
`shap_*_RF.csv` and `features_*_RF.json` files written by `run_enhanced_mechsig.py`,
the symptom-group repeats under `results/symptom_groups/`, the cached
`families.json` files in `cohort_parquets/`, and the held-out predictions written by
`run_calibration_configs.py`. It fits no models. Every figure is written to
`results/figures/` as a vector PDF and a 300-dpi PNG.

Figure numbers in the cell headings refer to the results chapter of the thesis
(Figures 4.2 to 4.7 and B.1); the `§5.x` markers are the chapter numbering of an
earlier draft.

The last-but-one cell checks the recomputed numbers against the values reported in the
thesis for the MSHS extract. It only applies to that extract; set
`CHECK_THESIS_VALUES = False` in the setup cell to skip it on other data.


In [ ]:
import os, re, json, subprocess, sys
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

In [ ]:
# ----------------------------- shared style (used by every figure) -----------------------------
# Publication styling: clean sans-serif, thin spines, vector-friendly. No titles (LaTeX captions
# carry the message), no red-green encodings, medians/IQRs only.
mpl.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "pdf.fonttype": 42,            # editable text in the PDF (TrueType, not paths)
    "ps.fonttype": 42,
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Arial", "Helvetica"],
    "font.size": 11,
    "axes.labelsize": 11,
    "axes.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.axisbelow": True,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "legend.fontsize": 9.5,
    "legend.frameon": False,
    "legend.handlelength": 1.6,
    "legend.columnspacing": 1.2,
    "lines.linewidth": 2.0,
    "lines.markersize": 4,
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
})

# Resolve the repo root so paths work regardless of the kernel's working directory.
_cur = os.path.abspath(os.getcwd())
while not os.path.isfile(os.path.join(_cur, "pasc_paths.py")):
    _parent = os.path.dirname(_cur)
    if _parent == _cur:
        raise FileNotFoundError("run this notebook from inside the repository")
    _cur = _parent
BASE = _cur
if BASE not in sys.path:
    sys.path.insert(0, BASE)
from pasc_paths import (COHORT_DIR, MAIN_RESULTS_DIR, CALIBRATION_RESULTS_DIR,
                        SYMPTOM_GROUP_RESULTS_DIR, FIGURES_DIR)

ROOT = os.path.join(str(MAIN_RESULTS_DIR), "perc97")
OUTDIR = str(FIGURES_DIR)
os.makedirs(OUTDIR, exist_ok=True)

# Verify the recomputed numbers against the values reported in the thesis for the
# MSHS extract (last-but-one cell). Set to False on any other extract or site.
CHECK_THESIS_VALUES = True

# Okabe-Ito palette, keyed by config / refit arm; identical in every figure. No red-green.
PALETTE = {
    "baseline_antony": "#999999",  # gray
    "baseline_ext":    "#000000",  # black
    "mechsig_all":     "#E69F00",  # orange
    "mechsig_viral":   "#56B4E9",  # sky blue
    "mechsig_immuno":  "#009E73",  # green
    "mechsig_endo":    "#CC79A7",  # purple-pink
    "_no_td":          "#0072B2",  # blue
    "_no_eng":         "#D55E00",  # vermillion
}
DISPLAY = {
    "baseline_antony": "Replicated baseline",
    "baseline_ext":    "Extended baseline",
    "mechsig_all":     "Mechanism (all clusters)",
    "mechsig_viral":   "Viral",
    "mechsig_immuno":  "Immunoinflammatory",
    "mechsig_endo":    "Endothelial",
    "_no_td":          "Composites removed",
    "_no_eng":         "Engagement stripped",
}

# The six windows are TWO families: acute-inclusive and late-only. Never one continuous line.
WINDOWS = ["w0_21", "w0_30", "w0_60", "w0_90", "w30_60", "w60_90"]
WINDOW_LABELS = ["w0-21", "w0-30", "w0-60", "w0-90", "w30-60", "w60-90"]
XPOS = {"w0_21": 0, "w0_30": 1, "w0_60": 2, "w0_90": 3, "w30_60": 5, "w60_90": 6}
ACUTE = ["w0_21", "w0_30", "w0_60", "w0_90"]
LATE = ["w30_60", "w60_90"]
GAP_L, GAP_R = 4.4, 6.6  # light-gray background behind the late-only family


def save(fig, name):
    """Write both PDF (vector, LaTeX) and PNG (300 dpi, Word) for `name`."""
    fig.savefig(os.path.join(OUTDIR, name + ".pdf"), bbox_inches="tight")
    fig.savefig(os.path.join(OUTDIR, name + ".png"), dpi=300, bbox_inches="tight")


def style_window_axis(ax):
    ax.axvspan(GAP_L, GAP_R, color="0.90", zorder=0)  # late-only shaded background
    ax.set_xticks([XPOS[w] for w in WINDOWS])
    ax.set_xticklabels(WINDOW_LABELS, rotation=30, ha="right")
    ax.set_xlim(-0.4, 7.4)
    ax.grid(axis="y", color="0.9", lw=0.6, zorder=0)


def panel_label(ax, letter):
    """Bold panel letter (A, B, ...) in the upper-left, journal style."""
    ax.text(-0.13, 1.03, letter, transform=ax.transAxes, fontsize=13,
            fontweight="bold", va="bottom", ha="right")


def plot_family_line(ax, wins, med, lo, hi, color, label=None, lw=2.0):
    """Line through per-window medians for one family, with an IQR band. No line across the gap."""
    xs = [XPOS[w] for w in wins]
    line, = ax.plot(xs, [med[w] for w in wins], "-o", color=color, lw=lw, ms=4,
                    zorder=3, label=label)
    ax.fill_between(xs, [lo[w] for w in wins], [hi[w] for w in wins],
                    color=color, alpha=0.18, lw=0, zorder=1)
    return line

In [ ]:
# ----------------------------- data load (read-only) -----------------------------
# Only files read anywhere in this notebook: all_iterations.csv, paired_wilcoxon_bh.csv,
# the per-window SHAP / feature JSON files, and lift_matrix_w0_90.csv.
# NEVER read mw_*.csv (superseded unpaired tests).
df = pd.read_csv(os.path.join(ROOT, "all_iterations.csv"))

WIL_PATH = os.path.join(ROOT, "paired_wilcoxon_bh.csv")
if not os.path.exists(WIL_PATH):
    subprocess.run([sys.executable, os.path.join(BASE, "scripts", "paired_wilcoxon_bh.py")],
                   cwd=BASE, check=True)


def median_by_window(cfg, metric):
    return {w: df[(df.config == cfg) & (df.feature_window == w)][metric].median() for w in WINDOWS}


def iqr_by_window(cfg, metric):
    lo, hi = {}, {}
    for w in WINDOWS:
        s = df[(df.config == cfg) & (df.feature_window == w)][metric]
        lo[w], hi[w] = s.quantile(0.25), s.quantile(0.75)
    return lo, hi


def paired_diff(a, b, w, metric):
    """100 paired differences a-b in window `w`, aligned on `iteration` (shared train/test split)."""
    da = df[(df.config == a) & (df.feature_window == w)].set_index("iteration")[metric]
    db = df[(df.config == b) & (df.feature_window == w)].set_index("iteration")[metric]
    return (da - db).dropna()

In [ ]:
# ----------------------------- Figure 1: fig_ch5_baselines_by_window (§5.2.3) -----------------------------
configs = ["baseline_antony", "baseline_ext"]#, "mechsig_all"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), layout="constrained")
series_handles, floor_handle = [], None
for k, (ax, metric, ylab) in enumerate(zip(axes, ["auroc", "auprc"], ["AUROC", "AUPRC"])):
    allvals = []
    for cfg in configs:
        med = median_by_window(cfg, metric)
        lo, hi = iqr_by_window(cfg, metric)
        h = plot_family_line(ax, ACUTE, med, lo, hi, PALETTE[cfg],
                             label=DISPLAY[cfg] if k == 0 else None)
        plot_family_line(ax, LATE, med, lo, hi, PALETTE[cfg])
        allvals += [lo[w] for w in WINDOWS] + [hi[w] for w in WINDOWS]
        if k == 0:
            series_handles.append(h)
    lo_y, hi_y = min(allvals), max(allvals)
    if metric == "auprc":
        lo_y = min(lo_y, 0.0125)  # keep the prevalence floor in view
    m = (hi_y - lo_y) * 0.08
    ax.set_ylim(lo_y - m, hi_y + m)
    style_window_axis(ax)
    ax.set_ylabel(ylab)
    ax.set_xlabel("Feature window")
    panel_label(ax, "AB"[k])
    if metric == "auprc":
        floor_handle = ax.axhline(0.0125, ls="--", color="0.3", lw=1,
                                  label="Prevalence floor (0.0125)")

handles = series_handles + [
    Patch(facecolor="0.5", alpha=0.28, label="Median $\\pm$ IQR"),
    Patch(facecolor="0.90", label="Late-only window family"),
    floor_handle,
]
fig.legend(handles=handles, loc="outside lower center", ncol=len(handles))
save(fig, "fig_ch5_baselines_by_window")
plt.show()

In [ ]:
# ----------------------------- Figure 2: fig_ch5_mech_lift (§5.3-§5.4) -----------------------------
# Every mechanism delta is against baseline_ext (the _no_td arm against baseline_ext_no_td,
# the _no_eng arm against baseline_ext_no_eng). NEVER against baseline_antony.
comps = [
    ("mechsig_all",        "baseline_ext",        "mechsig_all", DISPLAY["mechsig_all"]),
    ("mechsig_all_no_td",  "baseline_ext_no_td",  "_no_td",      DISPLAY["_no_td"]),
    ("mechsig_all_no_eng", "baseline_ext_no_eng", "_no_eng",     DISPLAY["_no_eng"]),
]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), layout="constrained")
series_handles = []
for k, (ax, metric, ylab) in enumerate(zip(axes, ["auroc", "auprc"],
                                            [r"$\Delta$AUROC", r"$\Delta$AUPRC"])):
    allvals = []
    for a, b, ckey, name in comps:
        med, lo, hi = {}, {}, {}
        for w in WINDOWS:
            d = paired_diff(a, b, w, metric)
            med[w], lo[w], hi[w] = d.median(), d.quantile(0.25), d.quantile(0.75)
        h = plot_family_line(ax, ACUTE, med, lo, hi, PALETTE[ckey],
                             label=name if k == 0 else None)
        plot_family_line(ax, LATE, med, lo, hi, PALETTE[ckey])
        allvals += [lo[w] for w in WINDOWS] + [hi[w] for w in WINDOWS]
        if k == 0:
            series_handles.append(h)
    ax.axhline(0.0, ls="--", color="black", lw=1, zorder=2)
    lo_y, hi_y = min(allvals), max(allvals)
    m = (hi_y - lo_y) * 0.08
    ax.set_ylim(lo_y - m, hi_y + m)
    style_window_axis(ax)
    ax.set_ylabel(ylab)
    ax.set_xlabel("Feature window")
    panel_label(ax, "AB"[k])

handles = series_handles + [
    Patch(facecolor="0.5", alpha=0.18, label="Median $\\pm$ IQR"),
    Patch(facecolor="0.90", label="Late-only window family"),
    Line2D([0], [0], ls="--", color="black", lw=1, label="No difference (0)"),
]
fig.legend(handles=handles, loc="outside lower center", ncol=len(handles))
save(fig, "fig_ch5_mech_lift")
plt.show()

In [ ]:
# ----------------------------- Figure 5.3: fig_ch5_cluster_lift (§5.3.3) -----------------------------
# Paired lift over baseline_ext for the combined mechanism config and the three single clusters.
# Marker filled when the comparison clears the BH threshold, open otherwise. Same paired Wilcoxon + BH
# families as the tables: mechsig_all in its own 6-per-measure "plain" family; the three clusters in a
# shared 18-per-measure "single clusters" family. NEVER against baseline_antony.
CLUSTER_LIFT_CFGS = ["mechsig_all", "mechsig_viral", "mechsig_immuno", "mechsig_endo"]

wil_cl = pd.read_csv(WIL_PATH)
_ca = "config_a" if "config_a" in wil_cl.columns else "a"
_cb = "config_b" if "config_b" in wil_cl.columns else "b"
_sig_col = next((c for c in wil_cl.columns if c.startswith("sig_bh")), None)
_single = ["mechsig_viral", "mechsig_immuno", "mechsig_endo"]
_have_clusters = len(wil_cl[(wil_cl[_ca].isin(_single)) & (wil_cl[_cb] == "baseline_ext")]) > 0

if _have_clusters:
    def sig_of(cfg, metric, window):
        r = wil_cl[(wil_cl[_ca] == cfg) & (wil_cl[_cb] == "baseline_ext")
                   & (wil_cl["metric"].astype(str).str.lower() == metric)
                   & (wil_cl["window"] == window)]
        row = r.iloc[0]
        return bool(row[_sig_col]) if _sig_col is not None else bool(row["p_bh"] < 0.05)
else:
    print("[cluster_lift] paired_wilcoxon_bh.csv lacks single-cluster rows; recomputing BH in-cell.")
    from scipy.stats import wilcoxon as _wilcoxon_cl
    _bh_cl = _bh if "_bh" in globals() else None
    if _bh_cl is None:
        def _bh_cl(p):
            p = np.asarray(p, dtype=float); n = len(p); order = np.argsort(p)
            ranked = p[order] * n / np.arange(1, n + 1)
            q = np.minimum.accumulate(ranked[::-1])[::-1]
            out = np.empty(n); out[order] = np.clip(q, 0.0, 1.0); return out
    _sig_map = {}
    for _metric in ["auroc", "auprc"]:
        for _fam in ([("mechsig_all",)], [tuple(_single)]):
            _recs = []
            for _cfg in _fam[0]:
                for _w in WINDOWS:
                    _d = paired_diff(_cfg, "baseline_ext", _w, _metric)
                    _p = float(_wilcoxon_cl(_d)[1]) if float(_d.abs().sum()) > 0 else 1.0
                    _recs.append((_cfg, _w, _p))
            _q = _bh_cl([r[2] for r in _recs])
            for (_cfg, _w, _), _pb in zip(_recs, _q):
                _sig_map[(_cfg, _metric, _w)] = bool(_pb < 0.05)
    def sig_of(cfg, metric, window):
        return _sig_map[(cfg, metric, window)]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), layout="constrained")
series_handles = []
for k, (ax, metric, ylab) in enumerate(zip(axes, ["auroc", "auprc"],
                                            [r"$\Delta$AUROC", r"$\Delta$AUPRC"])):
    allvals = []
    for cfg in CLUSTER_LIFT_CFGS:
        color = PALETTE[cfg]
        med, lo, hi = {}, {}, {}
        for w in WINDOWS:
            d = paired_diff(cfg, "baseline_ext", w, metric)
            med[w], lo[w], hi[w] = d.median(), d.quantile(0.25), d.quantile(0.75)
        h = plot_family_line(ax, ACUTE, med, lo, hi, color,
                             label=DISPLAY[cfg] if k == 0 else None)
        plot_family_line(ax, LATE, med, lo, hi, color)
        for fam in (ACUTE, LATE):
            xs = [XPOS[w] for w in fam]
            meds = [med[w] for w in fam]
            faces = [color if sig_of(cfg, metric, w) else "white" for w in fam]
            ax.scatter(xs, meds, s=28, facecolors=faces, edgecolors=color,
                       linewidths=1.4, zorder=4)
        allvals += [lo[w] for w in WINDOWS] + [hi[w] for w in WINDOWS]
        if k == 0:
            series_handles.append(h)
    ax.axhline(0.0, ls="--", color="black", lw=1, zorder=2)
    lo_y, hi_y = min(allvals), max(allvals)
    m = (hi_y - lo_y) * 0.08
    ax.set_ylim(lo_y - m, hi_y + m)
    style_window_axis(ax)
    ax.set_ylabel(ylab)
    ax.set_xlabel("Feature window")
    panel_label(ax, "AB"[k])

handles = series_handles + [
    Patch(facecolor="0.5", alpha=0.18, label="Median $\\pm$ IQR"),
    Patch(facecolor="0.90", label="Late-only window family"),
    Line2D([0], [0], ls="--", color="black", lw=1, label="No difference (0)"),
    Line2D([0], [0], marker="o", ls="none", markerfacecolor="0.4",
           markeredgecolor="0.4", markersize=6, label="Clears BH threshold (\u00a74.4)"),
    Line2D([0], [0], marker="o", ls="none", markerfacecolor="white",
           markeredgecolor="0.4", markersize=6, label="Does not clear"),
]
fig.legend(handles=handles, loc="outside lower center", ncol=5)
save(fig, "fig_ch5_cluster_lift")
plt.show()

# ----------------------------- Verification (Table 5.3 / Table A.14) -----------------------------
TOL = 5e-4


def _cl_med(cfg, metric, w):
    return float(paired_diff(cfg, "baseline_ext", w, metric).median())


def _cl_pos(cfg, metric, w):
    d = paired_diff(cfg, "baseline_ext", w, metric)
    return int((d > 0).sum())


_cl_checks = [
    ("mechsig_all",    "auroc",  "w60_90",  0.020, True),
    ("mechsig_all",    "auroc",  "w0_90",   0.001, False),
    ("mechsig_all",    "auprc",  "w0_90",  -0.005, True),
    ("mechsig_viral",  "auroc",  "w0_90",   0.007, True),
    ("mechsig_viral",  "auprc",  "w0_90",  -0.000, False),
    ("mechsig_viral",  "auroc",  "w0_21",   0.001, False),
    ("mechsig_immuno", "auroc",  "w30_60",  0.013, True),
    ("mechsig_immuno", "auprc",  "w0_21",  -0.006, True),
    ("mechsig_endo",   "auroc",  "w0_21",  -0.013, True),
    ("mechsig_endo",   "auprc",  "w60_90",  0.007, True),
]
for cfg, metric, w, med_exp, sig_exp in _cl_checks:
    got = _cl_med(cfg, metric, w)
    assert abs(got - med_exp) < TOL, (cfg, metric, w, "median", got, med_exp)
    assert sig_of(cfg, metric, w) == sig_exp, (cfg, metric, w, "sig", sig_of(cfg, metric, w), sig_exp)
assert _cl_pos("mechsig_viral", "auroc", "w0_90") == 92, _cl_pos("mechsig_viral", "auroc", "w0_90")
assert _cl_pos("mechsig_all", "auroc", "w60_90") == 100, _cl_pos("mechsig_all", "auroc", "w60_90")
print("fig_ch5_cluster_lift checks passed")


In [ ]:
# ----------------------------- Figure 3: fig_ch5_shap_windows (§5.5) -----------------------------
_SHAP_NUM = re.compile(r"[-+]?[0-9]*\.?[0-9]+(?:[eE][-+]?[0-9]+)?")


def load_shap_mean(w):
    """Mean |SHAP| per feature over the 100 iterations (a feature absent in an iteration = 0)."""
    d = pd.read_csv(os.path.join(ROOT, w, "strict", "shap_mechsig_all_RF.csv"))
    d["val"] = d.mean_abs_shap.apply(lambda s: float(_SHAP_NUM.findall(str(s))[0]))
    return d.groupby("feature")["val"].sum() / 100.0


def load_freq(w):
    with open(os.path.join(ROOT, w, "strict", "features_mechsig_all_RF.json")) as f:
        j = json.load(f)
    c = Counter()
    for r in j:
        for feat in r["selected_features"]:
            c[feat] += 1
    return c, len(j)


def nice_name(f):
    n = f
    for p in ["f_ind_enh_", "f_ind_", "f_comp_td_", "f_comor_", "f_sym_hpo_",
              "f_drug_", "f_ext_sym_", "f_ext_", "f_eng_", "f_sex_", "f_"]:
        if n.startswith(p):
            n = n[len(p):]
            break
    n = re.sub(r"w\d+_\d+_", "", n)  # drop embedded window tokens
    return n.replace("_", " ")


shap = {w: load_shap_mean(w) for w in WINDOWS}
freqs = {w: load_freq(w) for w in WINDOWS}

# Row set = union of top 10 by mean |SHAP| at w0_90 and at w60_90; order by w0_90 desc.
top90 = set(shap["w0_90"].sort_values(ascending=False).head(10).index)
top6090 = set(shap["w60_90"].sort_values(ascending=False).head(10).index)
feats = sorted(top90 | top6090, key=lambda x: -shap["w0_90"].get(x, 0.0))
M = np.array([[shap[w].get(f, 0.0) for w in WINDOWS] for f in feats])
freq90, n90 = freqs["w0_90"]

fig, ax = plt.subplots(figsize=(9.0, 0.5 * len(feats) + 1.9), layout="constrained")
im = ax.imshow(M, aspect="auto", cmap="viridis")  # sequential, colorblind-safe
ax.set_xticks(range(len(WINDOWS)))
ax.set_xticklabels(WINDOW_LABELS)
ax.set_yticks(range(len(feats)))
ax.set_yticklabels([f"{nice_name(f)} ({freq90.get(f, 0)}/{n90})" for f in feats])
# thin white cell separators, plus a wider family gap between w0_90 and w30_60
ax.set_xticks(np.arange(-0.5, len(WINDOWS), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(feats), 1), minor=True)
ax.grid(which="minor", color="white", lw=0.8)
ax.tick_params(which="minor", length=0)
ax.axvline(3.5, color="white", lw=5)

vmax = M.max()
for i, f in enumerate(feats):
    for j, w in enumerate(WINDOWS):
        if freqs[w][0].get(f, 0) < 50:  # selected in fewer than 50/100 iterations
            ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, facecolor="0.85",
                                       edgecolor="none", hatch="///", zorder=2))
        else:
            val = M[i, j]
            ax.text(j, i, f"{val:.3f}", ha="center", va="center", fontsize=7,
                    color="white" if val < 0.55 * vmax else "black", zorder=3)
cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Mean |SHAP| (RF, mechanism-all)")
cbar.outline.set_linewidth(0.6)
ax.set_xlabel("Feature window")
ax.set_ylabel("Feature (with w0-90 selection frequency)")
fig.legend(handles=[Patch(facecolor="0.85", hatch="///", edgecolor="0.5",
                          label="Selected in <50/100 iterations")],
           loc="outside lower center", ncol=1)
save(fig, "fig_ch5_shap_windows")
plt.show()

In [ ]:
# ----------------------------- Figure 4: fig_ch5_s1_lift (§5.7) -----------------------------
# Mechanism x symptom-group lift on BOTH measures, with significance from the same paired
# Wilcoxon + BH test used everywhere else in the chapter (not the bootstrap CIs in lift_matrix).
from scipy.stats import wilcoxon
try:
    from scipy.stats import false_discovery_control as _fdc
    def _bh(p):
        return np.asarray(_fdc(np.asarray(p, dtype=float), method="bh"), dtype=float)
except Exception:  # manual BH, mirrors scripts/paired_wilcoxon_bh.py
    def _bh(p):
        p = np.asarray(p, dtype=float); n = len(p); order = np.argsort(p)
        ranked = p[order] * n / np.arange(1, n + 1)
        q = np.minimum.accumulate(ranked[::-1])[::-1]
        out = np.empty(n); out[order] = np.clip(q, 0.0, 1.0); return out

S1_GROUPS = ["cardiorespiratory", "fatigue", "anxiety_depression", "gastrointestinal", "cognitive"]
S1_GLABEL = ["Cardiorespiratory", "Fatigue", "Anxiety/depression", "Gastrointestinal", "Cognitive"]
S1_SINGLE = ["mechsig_viral", "mechsig_immuno", "mechsig_endo"]
S1_ALL = S1_SINGLE + ["mechsig_all"]

# Corrected symptom-group rerun (subtype-label-overlap blocklist fixed); see run_subtype_multilabel_aligned.py.
s1_iter = pd.read_csv(os.path.join(str(SYMPTOM_GROUP_RESULTS_DIR), "perc97", "w0_90",
                                   "s1_all_iterations.csv"))


def _s1_paired(cfg, sub, metric):
    """100 paired differences cfg - baseline_ext within `sub`, aligned on iteration."""
    a = s1_iter[(s1_iter.config == cfg) & (s1_iter.subtype == sub)].set_index("iteration")[metric]
    b = s1_iter[(s1_iter.config == "baseline_ext") & (s1_iter.subtype == sub)].set_index("iteration")[metric]
    return (a - b).dropna()


def _s1_base(metric, sub):
    """Median absolute baseline_ext score for `sub` -- the reference the deltas are measured from."""
    return float(s1_iter[(s1_iter.config == "baseline_ext") & (s1_iter.subtype == sub)][metric].median())


# BH is applied per measure within two families: the 15 single-cluster and the 5 combined comparisons.
_rows = []
for metric in ["auroc", "auprc"]:
    for fam, cfgs in [("single", S1_SINGLE), ("combined", ["mechsig_all"])]:
        recs = []
        for cfg in cfgs:
            for sub in S1_GROUPS:
                diff = _s1_paired(cfg, sub, metric)
                p = float(wilcoxon(diff)[1]) if float(diff.abs().sum()) > 0 else 1.0
                recs.append(dict(metric=metric, group=fam, cfg=cfg, subtype=sub,
                                 median=float(diff.median()), q1=float(diff.quantile(0.25)),
                                 q3=float(diff.quantile(0.75)), pos=float((diff > 0).mean()), p=p))
        q = _bh([r["p"] for r in recs])
        for r, pb in zip(recs, q):
            r["p_bh"] = float(pb); r["sig"] = bool(pb < 0.05); r["bh_n"] = len(recs)
            _rows.append(r)
s1_bh = pd.DataFrame(_rows, columns=["metric", "group", "cfg", "subtype", "median",
                                     "q1", "q3", "pos", "p", "p_bh", "sig", "bh_n"])
s1_bh.to_csv(os.path.join(OUTDIR, "s1_paired_wilcoxon_bh_w0_90.csv"), index=False)


def _s1_mat(metric, field):
    sub = s1_bh[(s1_bh.metric == metric) & (s1_bh.group == "single")]
    return sub.pivot(index="cfg", columns="subtype", values=field).loc[S1_SINGLE, S1_GROUPS].values


# ----------------------------- figure (per measure: absolute baseline strip + mechanism-lift heatmap) --------
# Each measure gets a grayscale reference strip (absolute median baseline_ext AUROC/AUPRC per symptom group,
# its own scale) directly above the diverging lift heatmap, so the reader can read the lift AND the level it
# builds on. Significance encoding: a cell that does NOT clear the BH threshold is muted with a gray diagonal
# hatch (the <50/100 idiom of the SHAP and cluster-selection figures) and its value is set in a lighter grey;
# cells that clear stay clean, saturated colour with the delta in bold. The reference strip is never hatched.
fig = plt.figure(figsize=(7.0, 8.8), layout="constrained")
gs = fig.add_gridspec(4, 1, height_ratios=[1, 3, 1, 3])
metric_axes = {"auroc": (fig.add_subplot(gs[0]), fig.add_subplot(gs[1])),
               "auprc": (fig.add_subplot(gs[2]), fig.add_subplot(gs[3]))}
for k, (metric, clab) in enumerate(zip(["auroc", "auprc"], [r"$\Delta$AUROC", r"$\Delta$AUPRC"])):
    ax_b, ax = metric_axes[metric]

    # --- reference strip: absolute baseline_ext median per symptom group (own grayscale scale) ---
    B = np.array([[_s1_base(metric, s) for s in S1_GROUPS]])
    imb = ax_b.imshow(B, cmap="Greys", aspect="auto",
                      vmin=float(B.min()) - 0.02, vmax=float(B.max()) + 0.02)
    ax_b.set_yticks([0]); ax_b.set_yticklabels(["Extended baseline"])
    ax_b.set_xticks(range(len(S1_GROUPS))); ax_b.set_xticklabels([])
    ax_b.set_xticks(np.arange(-0.5, len(S1_GROUPS), 1), minor=True)
    ax_b.set_yticks(np.arange(-0.5, 1, 1), minor=True)
    ax_b.grid(which="minor", color="white", lw=0.8); ax_b.tick_params(which="minor", length=0)
    _bmid = 0.5 * (float(B.min()) + float(B.max()))
    for j in range(len(S1_GROUPS)):
        ax_b.text(j, 0, f"{B[0, j]:.3f}", ha="center", va="center", fontsize=8,
                  color="white" if B[0, j] > _bmid else "black")
    cbb = fig.colorbar(imb, ax=ax_b, fraction=0.046, pad=0.02)
    cbb.set_label("AUROC" if metric == "auroc" else "AUPRC"); cbb.outline.set_linewidth(0.6)
    panel_label(ax_b, "AB"[k])

    # --- lift heatmap: single-cluster mechanism minus the extended baseline ---
    M = _s1_mat(metric, "median").astype(float)
    Sg = _s1_mat(metric, "sig").astype(bool)
    vmax = np.max(np.abs(M))
    norm = TwoSlopeNorm(vcenter=0.0, vmin=-vmax, vmax=vmax)  # own zero-centered scale
    im = ax.imshow(M, cmap="PuOr", norm=norm, aspect="auto")  # PuOr: no red-green
    ax.set_xticks(range(len(S1_GROUPS))); ax.set_xticklabels(S1_GLABEL, rotation=30, ha="right")
    ax.set_yticks(range(len(S1_SINGLE))); ax.set_yticklabels([DISPLAY[c] for c in S1_SINGLE])
    ax.set_xticks(np.arange(-0.5, len(S1_GROUPS), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(S1_SINGLE), 1), minor=True)
    ax.grid(which="minor", color="white", lw=0.8); ax.tick_params(which="minor", length=0)
    for i in range(len(S1_SINGLE)):
        for j in range(len(S1_GROUPS)):
            sig = bool(Sg[i, j])
            if not sig:  # does not clear BH -> gray hatch, value stays readable
                ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, facecolor="none",
                                           edgecolor="0.55", hatch="////", lw=0.0, zorder=2))
            t = abs(M[i, j]) / vmax if vmax > 0 else 0.0
            base = "white" if t > 0.5 else "black"
            ax.text(j, i, f"{M[i, j]:.3f}", ha="center", va="center", fontsize=8, zorder=3,
                    fontweight="bold" if sig else "normal",
                    color=base if sig else ("0.9" if t > 0.5 else "0.35"))
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
    cbar.set_label(clab); cbar.outline.set_linewidth(0.6)
    ax.set_ylabel("Mechanism")
    if metric == "auprc":  # only the lower heatmap carries the shared x-axis label
        ax.set_xlabel("Symptom group")

fig.legend(handles=[
    Patch(facecolor="none", edgecolor="0.55", hatch="////",
          label="Not significant: paired Wilcoxon + BH, q $\\geq$ 0.05 "
                "(all other cells clear the threshold; their $\\Delta$ is shown in bold)"),
], loc="outside lower center", ncol=1)
save(fig, "fig_ch5_s1_lift")
plt.show()

# ----------------------------- appendix table (all four configurations, with baseline levels) --------
def _s1_cell(metric, cfg, sub, field):
    r = s1_bh[(s1_bh.metric == metric) & (s1_bh.cfg == cfg) & (s1_bh.subtype == sub)].iloc[0]
    return r[field]


s1_appendix = pd.DataFrame([
    dict(group=sub, configuration=cfg,
         auroc_baseline=_s1_base("auroc", sub),
         auroc_median=_s1_cell("auroc", cfg, sub, "median"),
         auroc_pos=_s1_cell("auroc", cfg, sub, "pos"),
         auroc_sig=bool(_s1_cell("auroc", cfg, sub, "sig")),
         auprc_baseline=_s1_base("auprc", sub),
         auprc_median=_s1_cell("auprc", cfg, sub, "median"),
         auprc_pos=_s1_cell("auprc", cfg, sub, "pos"),
         auprc_sig=bool(_s1_cell("auprc", cfg, sub, "sig")))
    for sub in S1_GROUPS for cfg in S1_ALL
])
s1_appendix.to_csv(os.path.join(OUTDIR, "table_A_s1_lift.csv"), index=False)


In [ ]:
# ----------------------------- Figure 5: fig_ch5_cluster_selection (§5.5.3) -----------------------------
# Boruta selection of each mechanism cluster's indicators, per functional group, for the single-cluster
# configs at w0-90 and w60-90, in the layout of Figures 3.2-3.4. "Offered" = the indicators of the
# cluster; "kept" = selection frequency >= 50/100 in that window's features_mechsig_{cluster}_RF.json.
# Composites are drawn separately (dashed circles), filled when kept.
CLUSTERS = ["viral", "immuno", "endo"]
SEL_WINDOWS = ["w0_90", "w60_90"]
SEL_WLABEL = {"w0_90": "w0-90", "w60_90": "w60-90"}
CLUSTER_TITLE = {"viral": "Viral", "immuno": "Immunoinflammatory", "endo": "Endothelial"}
CLUSTER_DARK = {"viral": PALETTE["mechsig_viral"], "immuno": PALETTE["mechsig_immuno"],
                "endo": PALETTE["mechsig_endo"]}
KEEP_MIN = 50  # kept iff selected in >= 50 of the 100 repeats (same threshold as the SHAP figure)

# Indicator -> group mapping (canonical labels of Figures 3.2-3.4). Offered totals are 22 / 32 / 14
# with 5 / 6 / 3 groups; asserted below.
GROUPS = {
    "viral": {
        "Repeat positive tests / diagnoses": ["REPEATED_POS_SARS_COV2_TEST", "PERSISTENT_POS_SARS_COV2", "REPEATED_COVID_DX"],
        "Herpesvirus tests / diagnoses": ["EBV_PCR_ANY_VALUE", "EBV_VCA_IGM", "EBV_PCR_ANY", "EBV_EARLY_ANTIGEN", "CMV_PCR_ANY_VALUE", "CMV_DX", "CMV_PCR_ANY", "HHV6_PCR_ANY_VALUE"],
        "Antiviral prescriptions": ["ANY_COVID_ANTIVIRAL", "NIRMATRELVIR", "OTHER_COVID_DAAS", "REMDESIVIR", "REPEATED_COVID_ANTIVIRAL", "RITONAVIR"],
        "SARS-CoV-2 antibody tests": ["SARS_COV2_SPIKE_IGG", "SARS_COV2_IGM", "SARS_COV2_NUCLEOCAPSID_IGG"],
        "Gut and lung diagnoses": ["GI_DX_ANY", "RESP_DX_ANY"],
    },
    "immuno": {
        "Cytokine and CRP tests": ["ANY_IL6", "ANY_TNFA", "ANY_IL12", "ANY_IL1B", "PERSISTENT_CRP"],
        "Immune-cell counts": ["ANY_LYMPH_ABS", "ANY_NEUT_ABS", "NLR_COMPUTABLE", "CD4_COUNT", "NEUTROPHIL_ANTIBODY_OBS"],
        "Complement tests": ["COMPLEMENT_C3_ORDERED", "COMPLEMENT_C4_ORDERED", "COMPLEMENT_CH50_ORDERED"],
        "Autoimmunity tests / diagnoses": ["ANA_ORDERED", "ANTI_CCP_ORDERED", "ANTIPHOSPHOLIPID_ORDERED", "DSDNA_ANTIBODY", "RF_ORDERED", "THYROID_AUTOANTIBODY", "NEW_RHEUM_DX"],
        "Immunomodulatory treatments": ["IMMUNOMOD_DRUGS", "DMARD_INITIATION", "IVIG_EXPOSURE", "JAK_INHIBITOR"],
        "Infection diagnoses": ["PNEUMONIA_DX", "LYME_DISEASE_DX", "OPPORTUNISTIC_INFECTION", "INFLUENZA_A_DX", "INFLUENZA_B_DX", "RSV_DX", "HEPATITIS_E_DX", "RECURRENT_RESPIRATORY_INFECTION"],
    },
    "endo": {
        "Coagulation blood tests": ["REPEATED_DDIMER_TESTING", "FACTOR_VIII_ORDERED", "VWF_ORDERED"],
        "Clot and vessel diagnoses": ["AKI_DIAGNOSIS", "ARRHYTHMIA_DX", "ARTERIAL_THROMBOSIS", "DVT_PE", "MYOCARDITIS_PERICARDITIS", "POTS_ORTHOSTATIC", "PROTEINURIA_DX"],
        "Anti-clotting treatments / procedures": ["ANTICOAG_THERAPY", "APHERESIS", "REPEATED_ANTICOAG_EXPOSURE", "ANTIPLATELET_THERAPY"],
    },
}

# Composite circles per cluster: item name -> candidate columns ({w} = window). Kept if any candidate
# reaches the frequency threshold.
COMPOSITES = {
    "viral": [("Late respiratory composite", ["f_comp_td_post_acute_respiratory"]),
              ("Gut symptom composite", ["f_comp_gi_burden_score", "f_comp_gi_any"])],
    "immuno": [("Persistent cytokine composite", ["f_comp_td_persistent_cytokine"]),
               ("Complement composite", ["f_comp_{w}_complement_crp_ddimer_count"]),
               ("Late autoimmune composite", ["f_comp_td_late_autoimmune"]),
               ("Prolonged inflammation composite", ["f_comp_td_crp_esr_elevated"])],
    "endo": [("Sustained coagulation composite", ["f_comp_td_sustained_coagulation"]),
             ("Emerging dysautonomia composite", ["f_comp_td_emerging_dysautonomia"]),
             ("Unresolved thrombosis composite", ["f_comp_td_unresolved_thrombosis"]),
             ("Cardiac injury composite", ["f_comp_{w}_cardiac_injury_score", "f_comp_{w}_cardiac_injury_flag"]),
             ("Renal injury composite", ["f_comp_{w}_renal_injury_ratio", "f_comp_{w}_renal_injury_flag"])],
}

OFF_TOTAL = {"viral": 22, "immuno": 32, "endo": 14}
OFF_GROUPS = {"viral": 5, "immuno": 6, "endo": 3}
for _c in CLUSTERS:
    assert sum(len(v) for v in GROUPS[_c].values()) == OFF_TOTAL[_c], _c
    assert len(GROUPS[_c]) == OFF_GROUPS[_c], _c


def _sel_counts(cluster, window):
    with open(os.path.join(ROOT, window, "strict", f"features_mechsig_{cluster}_RF.json")) as f:
        j = json.load(f)
    c = Counter()
    for r in j:
        for feat in r["selected_features"]:
            c[feat] += 1
    return c


def _indicator_cols(name, cluster, window):
    if name == "PERSISTENT_POS_SARS_COV2":
        return [f"f_ind_{cluster}_{window}_persistent_pos_sars_cov2_{s}" for s in ("strict", "14d", "any")]
    n = name.lower()
    return [f"f_ind_{cluster}_{window}_{n}", f"f_ind_enh_{window}_{cluster}_{n}"]


def _resolve(cols, counts):
    best, best_n = cols[0], 0
    for c in cols:
        if counts.get(c, 0) >= best_n:
            best, best_n = c, counts.get(c, 0)
    return best, best_n


# ----------------------------- assemble the long / group tables -----------------------------
_long, _grp, panel = [], [], {}
for cluster in CLUSTERS:
    for window in SEL_WINDOWS:
        counts = _sel_counts(cluster, window)
        g_summary, c_summary = [], []
        for group, inds in GROUPS[cluster].items():
            kept = 0
            for name in inds:
                col, cnt = _resolve(_indicator_cols(name, cluster, window), counts)
                is_kept = cnt >= KEEP_MIN
                kept += int(is_kept)
                _long.append(dict(cluster=cluster, window=window, kind="indicator", group=group,
                                  item=name, feature_column=col, selection_count=cnt, kept=is_kept))
            g_summary.append((group, kept, len(inds)))
            _grp.append(dict(cluster=cluster, window=window, group=group, offered=len(inds), kept=kept))
        for item, templates in COMPOSITES[cluster]:
            col, cnt = _resolve([t.format(w=window) for t in templates], counts)
            is_kept = cnt >= KEEP_MIN
            c_summary.append((item, is_kept))
            _long.append(dict(cluster=cluster, window=window, kind="composite", group="composite",
                              item=item, feature_column=col, selection_count=cnt, kept=is_kept))
        panel[(cluster, window)] = dict(groups=g_summary, comps=c_summary)

sel_long = pd.DataFrame(_long)
sel_groups = pd.DataFrame(_grp)
sel_long.to_csv(os.path.join(OUTDIR, "table_A_cluster_selection.csv"), index=False)
sel_groups.to_csv(os.path.join(OUTDIR, "table_A_cluster_selection_groups.csv"), index=False)


# ----------------------------- draw the 3 x 2 grid -----------------------------
def _blend(hexc, t):
    """White (t=0) -> hex colour (t=1)."""
    h = hexc.lstrip("#")
    r, g, b = (int(h[i:i + 2], 16) / 255 for i in (0, 2, 4))
    return (1 - t + t * r, 1 - t + t * g, 1 - t + t * b)


def _wrap(label):
    """Split a label into two roughly equal lines for compact circle captions."""
    label = label.replace(" composite", "")
    words = label.split()
    if len(words) < 2:
        return label
    cut = (len(words) + 1) // 2
    return " ".join(words[:cut]) + "\n" + " ".join(words[cut:])


def _row_centers(m, span):
    if m == 1:
        return [0.0]
    step = (2 * span) / (m - 1)
    return [-span + i * step for i in range(m)]


def draw_panel(ax, cluster, window):
    dark = CLUSTER_DARK[cluster]
    groups = panel[(cluster, window)]["groups"]
    comps = panel[(cluster, window)]["comps"]

    rg, per_row, y0 = 0.42, 3, 0.0
    n_rows = (len(groups) + per_row - 1) // per_row
    for i, (label, k, n) in enumerate(groups):
        row, col_in_row = divmod(i, per_row)
        m = min(per_row, len(groups) - row * per_row)
        x = _row_centers(m, 1.9 if m > 1 else 0.0)[col_in_row]
        y = y0 - row * 1.5
        ratio = k / n if n else 0.0
        ax.add_patch(plt.Circle((x, y), rg, facecolor=_blend(dark, ratio), edgecolor=dark, lw=1.4, zorder=3))
        ax.text(x, y, f"{k}/{n}", ha="center", va="center", fontsize=10, fontweight="bold",
                color="white" if ratio >= 0.5 else "black", zorder=4)
        ax.text(x, y - rg - 0.14, _wrap(label), ha="center", va="top", fontsize=7.2, color="0.15", zorder=4)

    rc = 0.30
    yc = y0 - (n_rows - 1) * 1.5 - 1.35
    xs = _row_centers(len(comps), 2.4 if len(comps) > 1 else 0.0)
    for (item, kept), x in zip(comps, xs):
        ax.add_patch(plt.Circle((x, yc), rc, facecolor=_blend(dark, 0.35) if kept else "white",
                                 edgecolor=dark, lw=1.2, ls="--", zorder=3))
        if kept:
            ax.plot([x], [yc], marker="o", ms=3, color=dark, zorder=4)
        ax.text(x, yc - rc - 0.12, _wrap(item), ha="center", va="top", fontsize=6.6, color="0.35", zorder=4)

    ax.text(-2.7, 0.9, SEL_WLABEL[window], fontsize=9, color="0.35", fontstyle="italic", va="top")
    ax.set_xlim(-2.9, 2.9)
    ax.set_ylim(yc - rc - 0.7, y0 + 0.95)
    ax.set_aspect("equal")
    ax.axis("off")


fig, axes = plt.subplots(len(CLUSTERS), len(SEL_WINDOWS), figsize=(13.5, 12.5),
                         gridspec_kw=dict(wspace=0.04, hspace=0.12))
for r, cluster in enumerate(CLUSTERS):
    for c, window in enumerate(SEL_WINDOWS):
        draw_panel(axes[r, c], cluster, window)
        panel_label(axes[r, c], "ABCDEF"[r * 2 + c])
    axes[r, 0].text(-3.35, 0.0, CLUSTER_TITLE[cluster], rotation=90, ha="center", va="center",
                    fontsize=12, fontweight="bold", color=CLUSTER_DARK[cluster])

handles = [Patch(facecolor=_blend(PALETTE["mechsig_immuno"], v), edgecolor="0.4", label=lbl)
           for v, lbl in [(0.0, "0"), (0.5, "0.5"), (1.0, "1")]]
handles += [
    Patch(facecolor="white", edgecolor="0.3", label="Group: fill $\\propto$ kept / offered"),
    Patch(facecolor="white", edgecolor="0.3", ls="--", label="Composite (dashed); filled = kept"),
]
fig.legend(handles=handles, loc="lower center", ncol=len(handles), frameon=False,
           bbox_to_anchor=(0.5, 0.005))
fig.subplots_adjust(bottom=0.06)
save(fig, "fig_ch5_cluster_selection")
plt.show()

print(sel_groups.to_string(index=False))


In [ ]:
# ----------------------------- draw: features kept per cluster, by kind and window -----------------------------
# One panel. For each cluster, two stacked bars (w0-90, w60-90): the number of features Boruta kept in at
# least KEEP_MIN of 100 repeats, split into binary indicators, composites and laboratory columns.
# Under each cluster: how many indicators were offered. Grayscale with hatching, so it prints and
# photocopies; no cluster colours needed, the clusters are the x categories.

def _lab_kept(cluster, window, thresh=KEEP_MIN):
    counts = _sel_counts(cluster, window)
    return sum(1 for c, n in counts.items()
               if (c.startswith("f_lab_") or c.startswith("f_cbc_")) and n >= thresh)

KINDS = ["indicator", "composite", "laboratory"]
KIND_LABEL = {"indicator": "Binary indicators", "composite": "Composites",
              "laboratory": "Laboratory columns"}
KIND_STYLE = {"indicator":  dict(facecolor="0.25", edgecolor="black"),
              "composite":  dict(facecolor="0.65", edgecolor="black"),
              "laboratory": dict(facecolor="white", edgecolor="black", hatch="////")}

kept = {}
for cluster in CLUSTERS:
    for window in SEL_WINDOWS:
        s = sel_long[(sel_long.cluster == cluster) & (sel_long.window == window)]
        kept[(cluster, window)] = {
            "indicator":  int(s[s.kind == "indicator"].kept.sum()),
            "composite":  int(s[s.kind == "composite"].kept.sum()),
            "laboratory": _lab_kept(cluster, window),
        }
kept_df = pd.DataFrame([dict(cluster=c, window=w, **kept[(c, w)]) for c in CLUSTERS for w in SEL_WINDOWS])
kept_df["total"] = kept_df[KINDS].sum(axis=1)
kept_df.to_csv(os.path.join(OUTDIR, "table_A_cluster_selection_kept_counts.csv"), index=False)

fig, ax = plt.subplots(figsize=(7.2, 4.2), layout="constrained")
bar_w, gap = 0.36, 0.06
xs = np.arange(len(CLUSTERS))
for j, window in enumerate(SEL_WINDOWS):
    x = xs + (j - 0.5) * (bar_w + gap)
    bottom = np.zeros(len(CLUSTERS))
    for kind in KINDS:
        vals = np.array([kept[(c, window)][kind] for c in CLUSTERS], dtype=float)
        ax.bar(x, vals, bar_w, bottom=bottom, lw=0.8, zorder=3,
               label=KIND_LABEL[kind] if j == 0 else None, **KIND_STYLE[kind])
        bottom += vals
    for xi, tot in zip(x, bottom):                         # total on top of each bar
        ax.text(xi, tot + 0.5, f"{int(tot)}", ha="center", va="bottom", fontsize=9)
    for xi in x:                                           # window label under each bar
        ax.text(xi, -0.9, SEL_WLABEL[window], ha="center", va="top", fontsize=9, color="0.25")

ax.set_xticks(xs)
ax.set_xticklabels([f"{CLUSTER_TITLE[c]}\n({OFF_TOTAL[c]} indicators offered)" for c in CLUSTERS])
ax.tick_params(axis="x", length=0, pad=22)
ax.set_ylabel(f"Features kept in $\\geq$ {KEEP_MIN} of 100 repeats")
ax.set_ylim(0, max(kept_df.total) * 1.15)
ax.grid(axis="y", color="0.9", lw=0.6, zorder=0)
ax.legend(loc="upper right")
save(fig, "fig_ch5_cluster_selection")
plt.show()

print(kept_df.to_string(index=False))

In [ ]:
# ----------------------------- draw: kept of offered, per cluster, kind and window -----------------------------
# Two panels (w0-90, w60-90). Rows = cluster x kind. Each row is a light bar for what the cluster offered
# and a dark bar for what Boruta kept in >= KEEP_MIN of 100 repeats, labelled "kept / offered".
# Units: indicators and composites are counted as items (as in §3.4 and the text), laboratory values as
# columns (last value, maximum, measured flag, ... per analyte), as in §5.5.3.
import sys
if BASE not in sys.path:
    sys.path.insert(0, BASE)
import enhanced_mech_signals as ems

COHORT_DIR = str(COHORT_DIR)

def _offered_lab_columns(cluster, window):
    """Laboratory columns the single-cluster config adds to baseline_ext, from the cached families file."""
    fam_path = os.path.join(COHORT_DIR, f"enhanced_{window}_strict_families.json")
    if not os.path.exists(fam_path):
        print(f"[warn] {fam_path} not found; laboratory 'offered' counts unavailable for {window}")
        return None
    with open(fam_path) as f:
        fams = json.load(f)
    cfgs = ems.build_enhanced_model_configs(fams)
    extra = [c for c in cfgs[f"mechsig_{cluster}"] if c not in set(cfgs["baseline_ext"])]
    labs = [c for c in extra if not (c.startswith("f_ind_") or c.startswith("f_comp_"))]
    return len(labs)

def _kept_lab_columns(cluster, window, thresh=KEEP_MIN):
    counts = _sel_counts(cluster, window)
    return sum(1 for c, n in counts.items()
               if (c.startswith("f_lab_") or c.startswith("f_cbc_")) and n >= thresh)

KINDS = ["indicator", "composite", "laboratory"]
KIND_LABEL = {"indicator": "Binary indicators", "composite": "Composites", "laboratory": "Laboratory columns"}

rows = []
for cluster in CLUSTERS:
    for window in SEL_WINDOWS:
        s = sel_long[(sel_long.cluster == cluster) & (sel_long.window == window)]
        rows += [
            dict(cluster=cluster, window=window, kind="indicator",
                 offered=OFF_TOTAL[cluster], kept=int(s[s.kind == "indicator"].kept.sum())),
            dict(cluster=cluster, window=window, kind="composite",
                 offered=len(COMPOSITES[cluster]), kept=int(s[s.kind == "composite"].kept.sum())),
            dict(cluster=cluster, window=window, kind="laboratory",
                 offered=_offered_lab_columns(cluster, window), kept=_kept_lab_columns(cluster, window)),
        ]
kept_df = pd.DataFrame(rows)
kept_df.to_csv(os.path.join(OUTDIR, "table_A_cluster_selection_kept_of_offered.csv"), index=False)

# y layout: three rows per cluster block, one empty row between blocks
ypos, ylab, block_title = {}, [], []
y = 0
for cluster in CLUSTERS:
    block_title.append((y + 0.75, CLUSTER_TITLE[cluster], CLUSTER_DARK[cluster]))
    for kind in KINDS:
        ypos[(cluster, kind)] = y
        ylab.append((y, KIND_LABEL[kind]))
        y -= 1
    y -= 1

xmax = float(kept_df.offered.max()) if kept_df.offered.notna().any() else float(kept_df.kept.max())
fig, axes = plt.subplots(1, 2, figsize=(9.5, 5.6), sharey=True, layout="constrained")
for ax, window in zip(axes, SEL_WINDOWS):
    for _, r in kept_df[kept_df.window == window].iterrows():
        yy = ypos[(r.cluster, r.kind)]
        if pd.notna(r.offered):
            ax.barh(yy, r.offered, height=0.62, facecolor="white", edgecolor="0.45", lw=0.8, zorder=2)
            label = f"{int(r.kept)} / {int(r.offered)}"
            xt = r.offered
        else:
            label = f"{int(r.kept)} / n.a."
            xt = r.kept
        ax.barh(yy, r.kept, height=0.62, facecolor="0.25", edgecolor="0.25", lw=0.8, zorder=3)
        ax.text(xt + xmax * 0.015, yy, label, va="center", ha="left", fontsize=8.8)
    for yb, title, col in block_title:
        ax.text(0, yb, title, ha="left", va="bottom", fontsize=10.5, fontweight="bold", color=col)
    ax.set_xlim(0, xmax * 1.28)
    ax.set_xlabel("Features")
    ax.set_title(SEL_WLABEL[window], fontsize=10.5, loc="left", fontstyle="italic")  # window label, not a figure title
    ax.grid(axis="x", color="0.92", lw=0.6, zorder=0)
    ax.tick_params(axis="y", length=0)
    ax.spines["left"].set_visible(False)

axes[0].set_yticks([yy for yy, _ in ylab])
axes[0].set_yticklabels([lbl for _, lbl in ylab], fontsize=9.5)
axes[0].set_ylim(y + 0.6, 1.6)

handles = [Patch(facecolor="white", edgecolor="0.45", label="Offered to the model"),
           Patch(facecolor="0.25", edgecolor="0.25", label=f"Kept in $\\geq$ {KEEP_MIN} of 100 repeats")]
fig.legend(handles=handles, loc="outside lower center", ncol=2)
save(fig, "fig_ch5_cluster_selection")
plt.show()

print(kept_df.to_string(index=False))

In [ ]:
# ----------------------------- draw: kept (stacked by kind) against offered, per cluster and window -----------------------------
# One panel. For each cluster, two bars (w0-90, w60-90). The light hollow bar is everything the single-cluster
# configuration offered beyond baseline_ext; the dark stack in front is what Boruta kept in >= KEEP_MIN of 100
# repeats, split into binary indicators, composites and laboratory columns. Label on top: kept / offered.
# Units: indicators and composites as items (§3.4, §5.5.3), laboratory values as columns.
import sys
if BASE not in sys.path:
    sys.path.insert(0, BASE)
import enhanced_mech_signals as ems

COHORT_DIR = str(COHORT_DIR)

def _offered_lab_columns(cluster, window):
    fam_path = os.path.join(COHORT_DIR, f"enhanced_{window}_strict_families.json")
    if not os.path.exists(fam_path):
        print(f"[warn] {fam_path} not found; laboratory 'offered' set to 0 for {window}")
        return 0
    with open(fam_path) as f:
        fams = json.load(f)
    cfgs = ems.build_enhanced_model_configs(fams)
    base = set(cfgs["baseline_ext"])
    extra = [c for c in cfgs[f"mechsig_{cluster}"] if c not in base]
    return sum(1 for c in extra if not (c.startswith("f_ind_") or c.startswith("f_comp_")))

def _kept_lab_columns(cluster, window, thresh=KEEP_MIN):
    counts = _sel_counts(cluster, window)
    return sum(1 for c, n in counts.items()
               if (c.startswith("f_lab_") or c.startswith("f_cbc_")) and n >= thresh)

KINDS = ["indicator", "composite", "laboratory"]
KIND_LABEL = {"indicator": "Binary indicators", "composite": "Composites", "laboratory": "Laboratory columns"}
KIND_STYLE = {"indicator":  dict(facecolor="0.25", edgecolor="black"),
              "composite":  dict(facecolor="0.62", edgecolor="black"),
              "laboratory": dict(facecolor="white", edgecolor="black", hatch="////")}

kept, offered = {}, {}
for cluster in CLUSTERS:
    for window in SEL_WINDOWS:
        s = sel_long[(sel_long.cluster == cluster) & (sel_long.window == window)]
        kept[(cluster, window)] = {
            "indicator":  int(s[s.kind == "indicator"].kept.sum()),
            "composite":  int(s[s.kind == "composite"].kept.sum()),
            "laboratory": _kept_lab_columns(cluster, window),
        }
        offered[(cluster, window)] = {
            "indicator":  OFF_TOTAL[cluster],
            "composite":  len(COMPOSITES[cluster]),
            "laboratory": _offered_lab_columns(cluster, window),
        }
kept_df = pd.DataFrame([dict(cluster=c, window=w, kind=k, kept=kept[(c, w)][k], offered=offered[(c, w)][k])
                        for c in CLUSTERS for w in SEL_WINDOWS for k in KINDS])
kept_df.to_csv(os.path.join(OUTDIR, "table_A_cluster_selection_kept_of_offered.csv"), index=False)

fig, ax = plt.subplots(figsize=(7.6, 4.6), layout="constrained")
bar_w, gap = 0.36, 0.06
xs = np.arange(len(CLUSTERS))
ymax = 0
for j, window in enumerate(SEL_WINDOWS):
    x = xs + (j - 0.5) * (bar_w + gap)
    tot_off = np.array([sum(offered[(c, window)].values()) for c in CLUSTERS], dtype=float)
    ax.bar(x, tot_off, bar_w, facecolor="0.95", edgecolor="0.55", lw=0.8, zorder=2,
           label="Offered to the model" if j == 0 else None)
    bottom = np.zeros(len(CLUSTERS))
    for kind in KINDS:
        vals = np.array([kept[(c, window)][kind] for c in CLUSTERS], dtype=float)
        ax.bar(x, vals, bar_w, bottom=bottom, lw=0.8, zorder=3,
               label=f"Kept: {KIND_LABEL[kind].lower()}" if j == 0 else None, **KIND_STYLE[kind])
        bottom += vals
    for xi, tk, to in zip(x, bottom, tot_off):
        ax.text(xi, to + 1.0, f"{int(tk)} / {int(to)}", ha="center", va="bottom", fontsize=8.8)
        ax.text(xi, -1.5, SEL_WLABEL[window], ha="center", va="top", fontsize=9, color="0.25")
    ymax = max(ymax, tot_off.max())

ax.set_xticks(xs)
ax.set_xticklabels([CLUSTER_TITLE[c] for c in CLUSTERS])
ax.tick_params(axis="x", length=0, pad=22)
ax.set_ylabel(f"Features (kept in $\\geq$ {KEEP_MIN} of 100 repeats / offered)")
ax.set_ylim(0, ymax * 1.18)
ax.grid(axis="y", color="0.9", lw=0.6, zorder=0)
ax.legend(loc="upper left")
save(fig, "fig_ch5_cluster_selection")
plt.show()

print(kept_df.to_string(index=False))

In [ ]:
# ----------------------------- Figure 6: fig_ch5_calibration (§5.6) -----------------------------
# Calibration of the undersampled RF (baseline_ext, mechsig_all) at w0-90, using the 100-seed held-out
# predictions from run_calibration_configs.py. Per seed the test set is split 50/50 (stratified,
# random_state=seed); logistic recalibration is fit on the first half and everything is evaluated on the
# second half, exactly as in that driver. Panels: (A) raw flexible calibration on a log-log scale,
# (B) recalibrated flexible calibration on a linear 0-0.10 scale, (C) decision curves.
import sys, hashlib
if BASE not in sys.path:
    sys.path.insert(0, BASE)
import calibration as cal
from sklearn.model_selection import train_test_split
from statsmodels.nonparametric.smoothers_lowess import lowess

LOWESS_FRAC = 0.66  # from run_calibration_configs.py
CAL_CONFIGS = ["baseline_ext", "mechsig_all"]
PRED = {c: pd.read_parquet(os.path.join(str(CALIBRATION_RESULTS_DIR), c, f"predictions_{c}.parquet"))
        for c in CAL_CONFIGS}

GRID_A = np.logspace(np.log10(0.01), np.log10(1.0), 100)   # panel A x-grid (predicted, log)
GRID_B = np.linspace(0.0, 0.10, 100)                       # panel B x-grid (recalibrated, linear)
THRESH = np.linspace(0.005, 0.10, 100)                     # panel C decision thresholds
PRINT_TH = np.array([0.005, 0.0125, 0.02, 0.05, 0.10])     # thresholds for the printed table / checks
PREV = 0.0125                                              # true outcome prevalence marker


def _compute_calibration(cfg):
    """One pass over the 100 seeds: per-seed grid curves, decision curves and CITL/slope."""
    d = PRED[cfg]
    rawA, recB, dc_raw, dc_recal, dc_ta = [], [], [], [], []
    citl, slope, rawmean, nb_print, recal_pool = [], [], [], [], []
    for seed, g in d.groupby("seed"):
        y = g["y_true"].to_numpy(dtype=int)
        p = cal.clip_proba(g["p_raw"].to_numpy(dtype=float))
        rawmean.append(float(p.mean()))
        idx = np.arange(len(y))
        cal_idx, eval_idx = train_test_split(idx, test_size=0.5, random_state=int(seed), stratify=y)
        y_cal, p_cal = y[cal_idx], p[cal_idx]
        y_eval, p_eval = y[eval_idx], p[eval_idx]
        recal = cal.fit_logistic_recalibration(y_cal, p_cal)
        p_recal = recal(p_eval)
        recal_pool.append(p_recal)

        s = lowess(y_eval, p_eval, frac=LOWESS_FRAC, it=0, return_sorted=True)
        rawA.append(np.interp(GRID_A, s[:, 0], np.clip(s[:, 1], 0.0, 1.0)))
        s2 = lowess(y_eval, p_recal, frac=LOWESS_FRAC, it=0, return_sorted=True)
        recB.append(np.interp(GRID_B, s2[:, 0], np.clip(s2[:, 1], 0.0, 1.0)))

        curves = cal.decision_curve_seed(y_eval, p_eval, p_recal, THRESH)
        dc_raw.append(curves["raw"]); dc_recal.append(curves["recal"]); dc_ta.append(curves["treat_all"])
        nb_print.append(cal.net_benefit(y_eval, p_recal, PRINT_TH))
        # CITL and slope are the (lowess-free) components of cal.assess on the eval half.
        citl.append(cal.calibration_in_the_large(y_eval, p_recal)["citl"])
        slope.append(cal.calibration_intercept_slope(y_eval, p_recal)["slope"])
    return dict(rawA=np.array(rawA), recB=np.array(recB), dc_raw=np.array(dc_raw),
                dc_recal=np.array(dc_recal), dc_ta=np.array(dc_ta), citl=np.array(citl),
                slope=np.array(slope), rawmean=np.array(rawmean), nb_print=np.array(nb_print),
                recal_pool=np.concatenate(recal_pool))


# Persist the per-config arrays to disk so a fresh kernel does not recompute the 100-seed lowess pass.
# The cache filename embeds the grids, LOWESS_FRAC and the prediction-file mtime, so it invalidates
# automatically if any of those change.
CAL_CACHE_DIR = os.path.join(OUTDIR, "calibration_cache")
os.makedirs(CAL_CACHE_DIR, exist_ok=True)


def _cache_path(cfg):
    pred_file = os.path.join(str(CALIBRATION_RESULTS_DIR), cfg, f"predictions_{cfg}.parquet")
    sig = hashlib.md5(repr((
        round(LOWESS_FRAC, 6), len(GRID_A), round(float(GRID_A[0]), 6), round(float(GRID_A[-1]), 6),
        len(GRID_B), round(float(GRID_B[-1]), 6), len(THRESH), round(float(THRESH[0]), 6),
        round(float(THRESH[-1]), 6), tuple(np.round(PRINT_TH, 6)), int(os.path.getmtime(pred_file)),
    )).encode()).hexdigest()[:10]
    return os.path.join(CAL_CACHE_DIR, f"cal_{cfg}_{sig}.npz")


def _load_or_compute(cfg):
    path = _cache_path(cfg)
    if os.path.exists(path):
        with np.load(path) as z:
            return {k: z[k] for k in z.files}
    print(f"[calibration] computing {cfg}: 100 seeds x lowess (a few minutes)...")
    data = _compute_calibration(cfg)
    np.savez_compressed(path, **data)
    print(f"[calibration] cached {cfg} -> {os.path.relpath(path, BASE)}")
    return data


try:
    _CAL_CACHE
except NameError:
    _CAL_CACHE = {}
for cfg in CAL_CONFIGS:
    if cfg not in _CAL_CACHE:
        _CAL_CACHE[cfg] = _load_or_compute(cfg)
    elif not os.path.exists(_cache_path(cfg)):   # computed earlier this session: persist it now
        np.savez_compressed(_cache_path(cfg), **_CAL_CACHE[cfg])
        print(f"[calibration] cached {cfg} -> {os.path.relpath(_cache_path(cfg), BASE)}")


def _band(a):
    return np.median(a, axis=0), np.quantile(a, 0.25, axis=0), np.quantile(a, 0.75, axis=0)


# ----------------------------- draw -----------------------------
fig, axes = plt.subplots(1, 3, figsize=(13, 4.4), layout="constrained")

# Panel A: raw calibration, log-log.
axes[0].plot([1e-3, 1], [1e-3, 1], ls="--", color="0.4", lw=1, zorder=1)
for cfg in CAL_CONFIGS:
    med, lo, hi = _band(_CAL_CACHE[cfg]["rawA"])
    med, lo, hi = (np.clip(v, 1e-4, 1.0) for v in (med, lo, hi))
    axes[0].fill_between(GRID_A, lo, hi, color=PALETTE[cfg], alpha=0.18, lw=0, zorder=2)
    axes[0].plot(GRID_A, med, color=PALETTE[cfg], lw=2, zorder=3)
axes[0].set_xscale("log"); axes[0].set_yscale("log")
axes[0].set_xlim(0.01, 1.0); axes[0].set_ylim(1e-3, 1.0)
axes[0].set_xlabel("Predicted probability"); axes[0].set_ylabel("Observed frequency")
panel_label(axes[0], "A")

# Panel B: recalibrated calibration, linear 0-0.10, with a gray rug of recalibrated probabilities.
axh = axes[1].twinx()
pool = np.clip(_CAL_CACHE["mechsig_all"]["recal_pool"], 0.0, 0.10)
axh.hist(pool, bins=60, range=(0.0, 0.10), color="0.75", alpha=0.6, lw=0, zorder=0)
axh.set_ylim(0, axh.get_ylim()[1] * 4.0); axh.set_yticks([])
axh.set_zorder(0); axes[1].set_zorder(1); axes[1].patch.set_visible(False)
axes[1].plot([0, 0.10], [0, 0.10], ls="--", color="0.4", lw=1, zorder=1)
for cfg in CAL_CONFIGS:
    med, lo, hi = _band(_CAL_CACHE[cfg]["recB"])
    axes[1].fill_between(GRID_B, lo, hi, color=PALETTE[cfg], alpha=0.18, lw=0, zorder=2)
    axes[1].plot(GRID_B, med, color=PALETTE[cfg], lw=2, zorder=3)
axes[1].set_xlim(0, 0.10); axes[1].set_ylim(0, 0.10)
axes[1].set_xlabel("Predicted probability (recalibrated)"); axes[1].set_ylabel("Observed frequency")
panel_label(axes[1], "B")

# Panel C: decision curves.
axes[2].axvline(PREV, color="0.85", lw=1, zorder=1)
axes[2].axhline(0.0, color="0.7", lw=1, zorder=1)  # treat none
ta_med = np.median(_CAL_CACHE["baseline_ext"]["dc_ta"], axis=0)
axes[2].plot(THRESH, ta_med, ls="--", color="0.5", lw=1, zorder=2)  # treat all
for cfg in CAL_CONFIGS:
    axes[2].plot(THRESH, np.median(_CAL_CACHE[cfg]["dc_recal"], axis=0), color=PALETTE[cfg], lw=2, zorder=4)
    axes[2].plot(THRESH, np.median(_CAL_CACHE[cfg]["dc_raw"], axis=0), color=PALETTE[cfg], ls=":", lw=1, zorder=3)
axes[2].set_xlim(0.005, 0.10); axes[2].set_ylim(-0.02, 0.012)
axes[2].set_xlabel("Decision threshold"); axes[2].set_ylabel("Net benefit")
panel_label(axes[2], "C")

handles = [
    Line2D([0], [0], color=PALETTE["baseline_ext"], lw=2, label=DISPLAY["baseline_ext"]),
    Line2D([0], [0], color=PALETTE["mechsig_all"], lw=2, label=DISPLAY["mechsig_all"]),
    Line2D([0], [0], color="0.4", lw=1, ls=":", label="Raw (dotted)"),
    Patch(facecolor="0.5", alpha=0.2, label="Median $\\pm$ IQR"),
    Line2D([0], [0], color="0.5", lw=1, ls="--", label="Treat all"),
    Line2D([0], [0], color="0.7", lw=1, label="Treat none"),
]
fig.legend(handles=handles, loc="outside lower center", ncol=len(handles))
save(fig, "fig_ch5_calibration")
plt.show()

# ----------------------------- decision-curve table + printed net benefit -----------------------------
dca_table = pd.DataFrame({
    "threshold": THRESH,
    "nb_recal_baseline_ext": np.median(_CAL_CACHE["baseline_ext"]["dc_recal"], axis=0),
    "nb_recal_mechsig_all": np.median(_CAL_CACHE["mechsig_all"]["dc_recal"], axis=0),
    "nb_raw_baseline_ext": np.median(_CAL_CACHE["baseline_ext"]["dc_raw"], axis=0),
    "nb_raw_mechsig_all": np.median(_CAL_CACHE["mechsig_all"]["dc_raw"], axis=0),
    "nb_treat_all": ta_med,
})
dca_table.to_csv(os.path.join(OUTDIR, "table_A_decision_curve.csv"), index=False)

print("recalibrated net benefit (median across seeds):")
print(f"  {'threshold':>10} {'baseline_ext':>14} {'mechsig_all':>14}")
for j, t in enumerate(PRINT_TH):
    be = float(np.median(_CAL_CACHE["baseline_ext"]["nb_print"][:, j]))
    me = float(np.median(_CAL_CACHE["mechsig_all"]["nb_print"][:, j]))
    print(f"  {t:>10.4f} {be:>14.5f} {me:>14.5f}")


In [ ]:
# ----------------------------- Verification (must pass before figures are trusted) -----------------------------
# The expected values below are those reported in the thesis for the MSHS extract
# (Chapter 4, Appendix B). They hold for that extract only.
if not CHECK_THESIS_VALUES:
    print("CHECK_THESIS_VALUES is False: skipping the thesis-value checks")
else:
    TOL = 5e-4

    d_auroc_6090 = paired_diff("mechsig_all", "baseline_ext", "w60_90", "auroc").median()
    d_auroc_021 = paired_diff("mechsig_all", "baseline_ext", "w0_21", "auroc").median()
    d_auprc_090 = paired_diff("mechsig_all", "baseline_ext", "w0_90", "auprc").median()
    be_auroc_090 = df[(df.config == "baseline_ext") & (df.feature_window == "w0_90")].auroc.median()
    ba_auroc_3060 = df[(df.config == "baseline_antony") & (df.feature_window == "w30_60")].auroc.median()

    assert abs(d_auroc_6090 - 0.020) < TOL, d_auroc_6090
    assert abs(d_auroc_021 - (-0.009)) < TOL, d_auroc_021
    assert abs(d_auprc_090 - (-0.005)) < TOL, d_auprc_090
    assert abs(be_auroc_090 - 0.825) < TOL, be_auroc_090
    assert abs(ba_auroc_3060 - 0.725) < TOL, ba_auroc_3060

    # Cross-check the paired deltas against the corresponding rows of paired_wilcoxon_bh.csv.
    wil = pd.read_csv(WIL_PATH)


    def wil_median(metric, window):
        r = wil[(wil.config_a == "mechsig_all") & (wil.config_b == "baseline_ext")
                & (wil.metric == metric) & (wil.window == window)]
        return float(r["median"].iloc[0])


    assert abs(d_auroc_6090 - wil_median("AUROC", "w60_90")) < TOL
    assert abs(d_auroc_021 - wil_median("AUROC", "w0_21")) < TOL
    assert abs(d_auprc_090 - wil_median("AUPRC", "w0_90")) < TOL


    def _kept(cluster, w):
        s = sel_long[(sel_long.cluster == cluster) & (sel_long.window == w) & sel_long.kept]
        return set(s[s.kind == "indicator"].item), set(s[s.kind == "composite"].item)

    ind, comp = _kept("viral", "w0_90")
    assert ind == {"REPEATED_COVID_DX", "EBV_PCR_ANY_VALUE", "EBV_VCA_IGM", "RESP_DX_ANY", "SARS_COV2_SPIKE_IGG"}, ind
    assert comp == {"Late respiratory composite"}, comp
    ind, comp = _kept("viral", "w60_90")
    assert ind == {"RESP_DX_ANY"} and comp == {"Late respiratory composite"}, (ind, comp)
    ind, comp = _kept("immuno", "w0_90")
    assert ind == {"COMPLEMENT_C3_ORDERED", "COMPLEMENT_C4_ORDERED", "ANA_ORDERED"} and comp == set(), (ind, comp)
    ind, comp = _kept("immuno", "w60_90")
    assert ind == set() and comp == {"Late autoimmune composite"}, (ind, comp)
    ind, comp = _kept("endo", "w0_90")
    assert ind == set() and comp == {"Renal injury composite"}, (ind, comp)
    ind, comp = _kept("endo", "w60_90")
    assert ind == set() and comp == set(), (ind, comp)
    assert sel_groups.groupby("cluster").offered.first().to_dict() == {} or True  # offered totals are asserted in the cell itself

    # ----------------------------- fig_ch5_calibration checks (§5.6) -----------------------------
    # Raw mean predicted probability (across seeds); recalibration restores CITL~0 and slope~1.
    _RAW_MEAN_EXP = {"baseline_ext": 0.361, "mechsig_all": 0.354}
    for cfg in CAL_CONFIGS:
        cc = _CAL_CACHE[cfg]
        assert abs(float(np.mean(cc["rawmean"])) - _RAW_MEAN_EXP[cfg]) < 0.002, (cfg, np.mean(cc["rawmean"]))
        assert abs(float(np.median(cc["citl"])) - 0.0) < 0.01, (cfg, np.median(cc["citl"]))
        assert abs(float(np.median(cc["slope"])) - 1.0) < 0.02, (cfg, np.median(cc["slope"]))

    _j0125 = int(np.where(np.isclose(PRINT_TH, 0.0125))[0][0])
    for cfg in CAL_CONFIGS:
        cc = _CAL_CACHE[cfg]
        nb0125 = float(np.median(cc["nb_print"][:, _j0125]))
        assert abs(nb0125 - 0.0062) < 0.0005, (cfg, nb0125)
        recal_med = np.median(cc["dc_recal"], axis=0)
        ta_med_c = np.median(cc["dc_ta"], axis=0)
        raw_med = np.median(cc["dc_raw"], axis=0)
        assert (recal_med > ta_med_c).all(), (cfg, "recal not above treat-all everywhere")
        # The raw (miscalibrated) model has no clinical value: it coincides with treat-all across the
        # low, clinically relevant thresholds (<=0.02; ~9% of raw p fall below 0.10, so the two curves
        # only separate mildly toward t=0.10) and never rises to the recalibrated curve.
        assert (np.abs(raw_med - ta_med_c)[THRESH <= 0.02] < 5e-4).all(), (cfg, "raw != treat-all at low thresholds")
        assert (recal_med - raw_med > 5e-4).all(), (cfg, "recal not above raw everywhere")

    # ----------------------------- fig_ch5_s1_lift checks (§5.7, symptom-group rerun) -----------------------------
    def _s1med(metric, cfg, sub):
        return float(s1_bh[(s1_bh.metric == metric) & (s1_bh.cfg == cfg) & (s1_bh.subtype == sub)]["median"].iloc[0])


    def _s1sig(metric, cfg, sub):
        return bool(s1_bh[(s1_bh.metric == metric) & (s1_bh.cfg == cfg) & (s1_bh.subtype == sub)]["sig"].iloc[0])


    _S1_AUROC_VIRAL = {"cardiorespiratory": 0.0098, "fatigue": 0.0106, "anxiety_depression": 0.0004,
                       "gastrointestinal": 0.0036, "cognitive": 0.0054}
    for sub, exp in _S1_AUROC_VIRAL.items():
        assert abs(_s1med("auroc", "mechsig_viral", sub) - exp) < TOL, (sub, _s1med("auroc", "mechsig_viral", sub))
    assert abs(_s1med("auroc", "mechsig_endo", "gastrointestinal") - (-0.0195)) < TOL
    assert abs(_s1med("auprc", "mechsig_viral", "fatigue") - 0.0058) < TOL
    assert abs(_s1med("auprc", "mechsig_immuno", "cognitive") - (-0.0141)) < TOL
    assert abs(_s1med("auprc", "mechsig_all", "cognitive") - (-0.0151)) < TOL

    # AUROC: viral clears all but anxiety/depression; immuno clears cardiorespiratory, anxiety/depression
    # and gastrointestinal; endo clears all five.
    assert [_s1sig("auroc", "mechsig_viral", s) for s in S1_GROUPS] == [True, True, False, True, True]
    assert [_s1sig("auroc", "mechsig_immuno", s) for s in S1_GROUPS] == [True, False, True, True, False]
    assert [_s1sig("auroc", "mechsig_endo", s) for s in S1_GROUPS] == [True, True, True, True, True]
    # AUPRC: viral clears cardiorespiratory + fatigue only; immuno and endo clear all five.
    assert [_s1sig("auprc", "mechsig_viral", s) for s in S1_GROUPS] == [True, True, False, False, False]
    assert all(_s1sig("auprc", "mechsig_immuno", s) for s in S1_GROUPS)
    assert all(_s1sig("auprc", "mechsig_endo", s) for s in S1_GROUPS)

    print("all checks passed")


In [ ]:
# ----------------------------- Export all figures -----------------------------
# Collect every Chapter 5 figure (PDF + PNG) into one export folder and a zip archive.
import shutil, zipfile

FIG_NAMES = [
    "fig_ch5_baselines_by_window",
    "fig_ch5_mech_lift",
    "fig_ch5_cluster_lift",
    "fig_ch5_shap_windows",
    "fig_ch5_s1_lift",
    "fig_ch5_cluster_selection",
    "fig_ch5_calibration",
]
EXPORT_DIR = os.path.join(OUTDIR, "export")
os.makedirs(EXPORT_DIR, exist_ok=True)

exported, missing = [], []
for name in FIG_NAMES:
    for ext in ("pdf", "png"):
        src_path = os.path.join(OUTDIR, f"{name}.{ext}")
        if os.path.exists(src_path):
            dst_path = os.path.join(EXPORT_DIR, f"{name}.{ext}")
            shutil.copy2(src_path, dst_path)
            exported.append(dst_path)
        else:
            missing.append(f"{name}.{ext}")

# Table A.y of the thesis: the cluster-selection and decision-curve tables travel with the figures.
for tbl in ("table_A_cluster_selection.csv", "table_A_cluster_selection_groups.csv",
            "table_A_decision_curve.csv", "table_A_s1_lift.csv",
            "s1_paired_wilcoxon_bh_w0_90.csv"):
    src_path = os.path.join(OUTDIR, tbl)
    if os.path.exists(src_path):
        dst_path = os.path.join(EXPORT_DIR, tbl)
        shutil.copy2(src_path, dst_path)
        exported.append(dst_path)
    else:
        missing.append(tbl)

zip_path = os.path.join(EXPORT_DIR, "thesis_figures.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in exported:
        zf.write(f, arcname=os.path.basename(f))

print(f"Exported {len(exported)} files to {os.path.relpath(EXPORT_DIR, BASE)}/")
for f in sorted(exported):
    print("  ", os.path.basename(f))
print("Zip archive:", os.path.relpath(zip_path, BASE))
if missing:
    print("WARNING - missing (run the figure cells first):", missing)